# 00 · Project setup and experiment contract

**Single responsibility:** validate the environment, configuration, reproducibility controls, and ESP32 constraints

Run after the preceding numbered notebook unless the inputs already exist. Every generated artifact is written outside the notebook so this stage is reproducible.


## Why this design

The target is image-level **person / no-person** gating, not bounding-box detection. The positive-label rule follows the Visual Wake Words paper: a non-crowd person box must occupy at least 0.5% of image area. The deployment baseline is RGB 96×96, MobileNetV1-style depthwise convolutions, and full INT8 quantization.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/base.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from vww_esp32.config import load_config, resolve_paths, seed_everything

config, ROOT = load_config(ROOT / 'configs/base.yaml')
paths = resolve_paths(config, ROOT)
seed_everything(config['project']['seed'])
ROOT


PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32')

In [2]:
import platform
import numpy as np
import pandas as pd

versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__}
try:
    import tensorflow as tf
    versions['tensorflow'] = tf.__version__
    versions['accelerators'] = [device.name for device in tf.config.list_physical_devices() if device.device_type != 'CPU']
except ImportError as exc:
    versions['tensorflow_error'] = str(exc)
versions

{'python': '3.10.13',
 'numpy': '2.1.3',
 'pandas': '2.3.3',
 'tensorflow': '2.20.0',
 'accelerators': []}

In [3]:
required_sections = {'project', 'paths', 'data', 'preprocessing', 'model', 'training', 'evaluation', 'export'}
missing = required_sections - config.keys()
assert not missing, f'Missing config sections: {missing}'
assert config['preprocessing']['image_size'] == [96, 96]
assert 0 < config['data']['min_person_area_fraction'] < 1
config

{'project': {'name': 'visual-wake-words-esp32-cam', 'seed': 42},
 'paths': {'raw': 'data/raw/coco2017',
  'interim': 'data/interim',
  'processed': 'data/processed',
  'artifacts': 'artifacts'},
 'data': {'dataset': 'MS COCO 2017',
  'annotation_url': 'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
  'target_category': 'person',
  'min_person_area_fraction': 0.005,
  'validation_fraction': 0.15,
  'max_samples': {'train': 12000, 'val': 2000, 'test': 2000},
  'balance_training_classes': True,
  'download_workers': 12,
  'download_timeout_seconds': 45},
 'preprocessing': {'image_size': [96, 96],
  'channels': 3,
  'resize_method': 'bilinear',
  'batch_size': 64,
  'shuffle_buffer': 4096,
  'cache': False},
 'model': {'architecture': 'tiny_mobilenet_v1',
  'width_multiplier': 0.25,
  'dropout': 0.1,
  'l2': 1e-05},
 'training': {'epochs': 35,
  'learning_rate': 0.001,
  'label_smoothing': 0.05,
  'early_stopping_patience': 7,
  'reduce_lr_patience': 3,
  'monito

In [4]:
disk_target_gb = 8  # conservative budget for the default selected-image experiment
import shutil
free_gb = shutil.disk_usage(ROOT).free / 1024**3
print(f'Free disk: {free_gb:.1f} GiB')
if free_gb < disk_target_gb:
    print('WARNING: lower data.max_samples before acquisition or free disk space.')

Free disk: 457.2 GiB


## Experiment contract

- Tune architecture and threshold with `train` + `val` only.
- Open the held-out COCO-derived `test` labels only in notebook 06.
- Report PR-AUC, recall, specificity, F1, calibration, model bytes, and desktop INT8 parity.
- Release only after device-captured validation, measured latency, and measured tensor-arena high-water usage.